<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Segmentation/SAM3.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# SAM3 Local Demo 


https://github.com/facebookresearch/sam3

"SAM 3 is a unified foundation model for promptable segmentation in images and videos. It can detect, segment, and track objects using text or visual prompts such as points, boxes, and masks."

This notebook demonstrates **official SAM3** usage with a **local checkpoint**:

- local weights: `weights/sam3.pt`
- local data: files under `data/`

It includes:
- Open-set text detection (free prompt list)
3. Segmentation masks
4. Geometric prompting (box prompt)
5. Video tracking
6. Point-prompt refinement for tracking

### Weights download

Download weights from this link: https://modelscope.cn/models/facebook/sam3/file/view/master/sam3.pt?status=2

and put it into *weights folder*

In [ ]:
import os
import torch

CKPT = 'weights/sam3.pt'
DATA = 'data'

assert os.path.exists(CKPT), f'Missing checkpoint: {CKPT}'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE != 'cuda':
    raise RuntimeError('CUDA is required for this official SAM3 build path in this environment.')

print(f'Using device: {DEVICE}')


## 1) Pick Local Samples (Image + Video Frames)

Few helpers for displaying results

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageOps, ImageFont
import colorsys
from matplotlib import colors as mcolors


color_names = list(mcolors.CSS4_COLORS.keys())  # list[str]

#============================= Helper =======================================
def _label_color(label, default='red'):
    if type(label) is  int:
        idx = label % len(color_names)
        return color_names[idx]
    
    if label == "GT":
        return 'lime'
    return default


def draw_boxes(image, boxes, labels=None, color='red', width=8, size=48):
    out = image.copy()
    dr = ImageDraw.Draw(out)
    font = ImageFont.truetype("DejaVuSans.ttf", size=size)
    labels = labels or [''] * len(boxes)
    for b, t in zip(boxes, labels):
        x1, y1, x2, y2 = [float(v) for v in b]
        c = color
        if t:
            c = _label_color(t, default=color)
            dr.text((x1 + 3, max(0, y1 - 12)), str(t), fill=c, font=font)
        dr.rectangle([x1, y1, x2, y2], outline=c, width=width)
    return out
#============================= End Helper =======================================

image_path = Path('data/MOT17-09/img1/000001.jpg')
label_path = Path('data/MOT17-09/gt/gt.txt')

image = Image.open(image_path).convert('RGB')
gt_rows = [
    row.split(',')
    for row in label_path.read_text().splitlines()
    if row.strip() and int(float(row.split(',')[0])) == 1
]
gt_boxes = [
    [float(x), float(y), float(x) + float(w), float(y) + float(h)]
    for _, _, x, y, w, h, *_ in gt_rows
]

print('Image sample:', image_path)
print('Label file :', label_path)
print('GT boxes   :', len(gt_boxes))

img_gt = draw_boxes(image, gt_boxes, color='lime', width = 4, size=24)
img_gt



We see: GT isn't perfect :(

## Build SAM3 Image Model (Local Weights Only)

In [ ]:
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor



image_model = build_sam3_image_model(
    checkpoint_path=str(CKPT),
    load_from_HF=False,
    device=DEVICE,
    eval_mode=True,
    enable_segmentation=True,
)
image_processor = Sam3Processor(
    model=image_model,
    resolution=1008,
    device=DEVICE,
    confidence_threshold=0.2,
)
print('Image model ready')


## Open-Set Detection 

### Find all persons

In [ ]:
state = image_processor.set_image(image)
output = image_processor.set_text_prompt(
    prompt='person', # Prompt in one string
    state=state
    )

masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

draw_boxes(img_gt,boxes,width=4,color='red')

### More sophisticated prompt

In [ ]:
output = image_processor.set_text_prompt( 
    prompt='Company logo with red background',
     state=state
     )
masks, boxes, scores = output["masks"], output["boxes"], output["scores"]
draw_boxes(image,boxes,labels=scores.tolist(),size =22)

### Filter by threshold


In [ ]:
thr = 0.6
keep = output['scores'] > thr

boxes_thr = output['boxes'][keep]
scores_thr = output['scores'][keep]
masks_thr = output['masks'][keep]

vis = draw_boxes(image, boxes_thr, color='red',size=18)
vis


now we see oly one bbox

##  Video (MOT17-09)


Traking objects in video

In [ ]:

# ============================= Visualization helper =======================================
def make_grid(images, cols=3, pad=8, bg=(24, 24, 24)):
    if len(images) == 0:
        return Image.new('RGB', (320, 120), color=bg)
    cell_w = max(im.width for im in images)
    cell_h = max(im.height for im in images)
    rows = (len(images) + cols - 1) // cols
    grid = Image.new('RGB', (cols * cell_w + pad * (cols + 1), rows * cell_h + pad * (rows + 1)), color=bg)
    for i, im in enumerate(images):
        r, c = divmod(i, cols)
        x = pad + c * (cell_w + pad)
        y = pad + r * (cell_h + pad)
        grid.paste(ImageOps.fit(im, (cell_w, cell_h)), (x, y))
    return grid
# ============================= End Visualization helper =======================================


# Load video frames

# for reduce time consumption we use only every 10-th frame
frame_paths = sorted(Path('data/MOT17-09/img1').glob('*.jpg'))[::10]
video_frames = [Image.open(p).convert('RGB') for p in frame_paths]

print('Frames:', len(video_frames))
print('First :', frame_paths[0].name)
print('Last  :', frame_paths[-1].name)

# For visualization use 12 frame with step 3 (10*3 = 30)
make_grid(video_frames[::3][:12], cols=3)



## Build SAM3 Video Model
 using local weights

In [ ]:
from sam3.model_builder import build_sam3_video_model

video_model = build_sam3_video_model(
    checkpoint_path=str(CKPT),
    load_from_HF=False,
    device="cuda", #'CUDA is required for SAM3 video model.'
)
print('Video model ready')


##  Track Persons + Threshold


In [ ]:
# Pass data into model

inference_state = video_model.init_state(
    resource_path=video_frames, 
    # Alternatively you can pass path to mp4 of folder with jpeg's resource_path="data/MOT17-09/img1", 
    offload_video_to_cpu=True, # where decoded video/frame tensors are stored
)
prompt = 'person'  # add prompt here
_ = video_model.add_prompt(inference_state, frame_idx=0, text_str=prompt)

tracked = {}

propagate_in_video_iterator =video_model.propagate_in_video(
    inference_state,
    start_frame_idx=0,
    max_frame_num_to_track=len(video_frames),
    reverse=False,
)

for i, out in propagate_in_video_iterator:
    tracked[i] = out 

print('Tracked frames:', len(tracked))



Show track result. Object with same ID must have same color

In [ ]:
thr = 0.2
vis = []

for i, frame in enumerate(video_frames[:12]):
    out = tracked.get(i)
    v = frame.copy()
    if out is not None:       
        boxes = []
        labels = []
        boxes_xywh = out.get('out_boxes_xywh')
        probs = out.get('out_probs')
        obj_ids = out.get('out_obj_ids', [])

        if boxes_xywh is not None:
            H, W = frame.height, frame.width
            for j, b in enumerate(boxes_xywh):
                p = float(probs[j]) if probs is not None and j < len(probs) else 1.0
                if p < thr:
                    continue
                x, y, w, h = b
                boxes.append([x * W, y * H, (x + w) * W, (y + h) * H])
                oid = int(obj_ids[j]) if j < len(obj_ids) else j
                labels.append(f'{oid}')

        v = draw_boxes(v, boxes, labels=labels)
    vis.append(v)

make_grid(vis, cols=3)


## Segmentation in video

In [ ]:
oid = 1  # segmented object id (woman at the right side)
vis = []
for i, im in enumerate(video_frames[:12]):
    out = tracked[i]
    ids = list(out['out_obj_ids'])
    if oid not in ids:
        vis.append(im)
        continue
    j = ids.index(oid)
    m = out['out_binary_masks'][j].astype('uint8') * 255
    m = Image.fromarray(m, 'L')
    red = Image.blend(im, Image.new('RGB', im.size, 'red'), .4) # semi-transparent red mask
    vis.append(Image.composite(red, im, m))
make_grid(vis, cols=3)


Prompt-only segmentation example

In [ ]:

inference_state  = video_model.init_state(resource_path=video_frames)
_ = video_model.add_prompt(inference_state , frame_idx=0, text_str='blue t-shirt') # prompt
iter = video_model.propagate_in_video(
    inference_state ,
    start_frame_idx=0,
    max_frame_num_to_track=len(video_frames),
    offload_video_to_cpu=True,
    )
vis = []
for i, out in iter:
    im = video_frames[i]
    if i % 5  != 0:
        continue
    
    if len(out['out_obj_ids']) == 0:
        m = Image.new('L', im.size, 0)
    else:
        m = Image.fromarray(out['out_binary_masks'][0].astype('uint8') * 255, 'L')
    red = Image.blend(im, Image.new('RGB', im.size, 'red'), .4)
    vis.append(Image.composite(red, im, m))
make_grid(vis, cols=3)


## Notes

- `output['scores'] > thr` is the simplest confidence filter pattern.
